#### data -- LCB and MCLM
load


In [ ]:
import regex
import pandas as pd 
from datasets import load_dataset

LANGS = ["en", "ne", "hi", "ru", "de"]
ds = load_dataset("amphora/MCLM", "MT-MATH100", split="test")
df = ds.to_pandas()
print(ds)
print(df.shape)
df.loc[0, LANGS + ["answer"]]

masking math and code (we need this masked version for script counting, the model sees the full prompt). we should not count this math symbols as language switch, so we are making note of that here.

In [ ]:
MATH = regex.compile(r"\$\$.*?\$\$|\$.*?\$|\\\(.*?\\\)|\\\[.*?\\\]", flags=regex.S)
ASY = regex.compile(r"\[asy\].*?\[/asy\]", flags=regex.S)

def strip_math(s: str, placeholder: str = " ") -> str:
    """Replace LaTeX math spans and [asy] code blocks with `placeholder` (" " for counting; "(M)" only for viewing)."""
    return ASY.sub(placeholder, MATH.sub(placeholder, s))

print(strip_math(df.loc[0, "ne"], placeholder="(M)"))   # viewing only

In [ ]:
def letter_scripts(s: str) -> dict[str, int]:
    """Count letters per script, outside math/code."""
    s = strip_math(s)
    return {sc: len(regex.findall(rf"\p{{{sc}}}", s)) for sc in ["Latin", "Devanagari", "Cyrillic", "Han"]}

stats = pd.DataFrame({
    "chars_ne": df["ne"].str.len(),
    "has_asy": df["en"].str.contains(r"\[asy\]"),
    "n_dollar": df["en"].str.count(r"\$"),
})
print(stats.describe(), "\nproblems with [asy]:", stats.has_asy.sum())

for lang in LANGS:
    sc = pd.DataFrame([letter_scripts(s) for s in df[lang]])
    share = (sc.sum() / sc.sum().sum()).round(3)
    print(lang, share.to_dict(), "| prompts with Latin outside math:", (sc.Latin > 0).sum())

for i, s in enumerate(df["ne"]):
    words = regex.findall(r"\p{Latin}+", strip_math(s))
    if words:
        print(i, words[:10])

load LCB Dolly data

In [ ]:
import io
import urllib.request
import zipfile

LCB_URL = "https://github.com/for-ai/language-confusion/raw/main/test_sets.zip"
with urllib.request.urlopen(LCB_URL) as r:
    zf = zipfile.ZipFile(io.BytesIO(r.read()))

lcb = pd.DataFrame({lang: pd.read_csv(zf.open(f"test_sets/monolingual/dolly/{lang}.csv"))["prompt"]
                    for lang in ["hi", "ru", "es"]})
print(lcb.shape)
lcb.loc[[0, 60, 84, 92]]

token counts; how long, how fast will be the generation

In [ ]:
from transformers import AutoTokenizer

tok = AutoTokenizer.from_pretrained("Qwen/Qwen3-0.6B")

def n_tokens(s: str) -> int:
    return len(tok(s, add_special_tokens=False)["input_ids"])

mt_len = df[["en", "ne", "hi", "ru"]].map(n_tokens)
print(mt_len.describe().round(1))
print("median ratio vs en:\n", mt_len[["ne", "hi", "ru"]].div(mt_len.en, axis=0).median().round(2))

lcb_len = lcb.map(n_tokens)
print(lcb_len.describe().round(1))

pick 20 parallel LCB prompts to translate to nepali(no lcb data for nepali)

In [ ]:
ALIGNED = lcb.index[:85]                                      
has_latin = lcb.loc[ALIGNED].map(lambda s: bool(regex.search(r"\p{Latin}", s)))
ok = ALIGNED[~(has_latin.hi | has_latin.ru)]                  
pick = (lcb_len.loc[ok, "hi"] + lcb_len.loc[ok, "ru"]).sort_values().index[:20]
lcb.loc[pick, ["hi", "ru", "es"]]

Nepali LCB prompts

In [ ]:
# Nepali translations of LCB Dolly rows (from the Hindi; draft by Claude, checked by me on 2026-10-30)
NE_LCB = {
    64: "अजगर भनेको के हो?",
    75: "कतिवटा ग्रहहरू छन्?",
    60: "के कला बेकार हो?",
    81: "राम्रो तस्बिर कस्तो हुन्छ?",
    59: "चन्द्रमामा बस्दा कस्तो हुन्छ?",
    67: "सफ्टवेयर शिल्पकला भनेको के हो?",
    62: "मैले कस्तो किसिमको डुङ्गा किन्नुपर्छ?",
    69: "म दौडन कसरी सुरु गरूँ?",
    0:  "स्याउ खाने 5 तरिकाहरू के-के हुन्?",
    5:  "मलाई 5 वटा डरलाग्दा फिल्महरू सुझाउनुहोस्।",
    49: "तपाईं यात्राको योजना कसरी बनाउनुहुन्छ?",
    9:  "जनावरहरूले सञ्चार गर्ने 5 तरिकाहरू",
    74: "मैले आफ्नो पैसा कसरी लगानी गर्नुपर्छ?",
    79: "मानिसहरू बगैंचा लगाउन किन मन पराउँछन्?",
    57: "के सामाजिक सञ्जाल तपाईंका लागि राम्रो हो?",
    22: "पौडी खेल्ने केही सामान्य तरिकाहरू के-के हुन्?",
    45: "के म घरमै चकलेट बनाउन सक्छु?",
    82: "भान्सेहरू कोशर नुन किन प्रयोग गर्छन्?",
    65: "ट्रान्जिस्टरले सिग्नललाई कसरी बढाउँछ?",
    83: "मौसमको पूर्वानुमान कसरी गरिन्छ?",
}

lcb_pilot = lcb.loc[pick, ["hi", "ru", "es"]].copy()
lcb_pilot["ne"] = pd.Series(NE_LCB)            # aligns on the row index, not on order
assert lcb_pilot["ne"].notna().all(), "a picked row has no Nepali translation"

# sanity: Nepali has no Latin letters, and its token count is close to Hindi's
print(lcb_pilot["ne"].map(lambda s: bool(regex.search(r"\p{Latin}", s))).sum(), "rows with Latin")
print(lcb_pilot[["hi", "ne", "ru"]].map(n_tokens).median())
lcb_pilot[["hi", "ne", "ru", "es"]]

pick 20 MCLM problems: no [asy], no Latin outside math in ne/hi/ru, not too long; random (seed 0) rather than shortest, so it's not only the easiest problems

In [ ]:
PILOT_LANGS = ["ne", "hi", "ru"]
MAX_EN_TOKENS = 120                                            # keeps prompts (and likely traces) short

no_latin = pd.concat([df[l].map(lambda s: not regex.search(r"\p{Latin}", strip_math(s))) for l in PILOT_LANGS], axis=1).all(axis=1)
mt_ok = df.index[~stats.has_asy & no_latin & (mt_len.en <= MAX_EN_TOKENS)]
mt_pick = pd.Series(mt_ok).sample(20, random_state=0).sort_values().to_list()
print(len(mt_ok), "eligible →", mt_pick)
print(mt_len.loc[mt_pick].median())
df.loc[mt_pick, ["en", "ne", "answer"]]

save both pilot prompt sets in long format (one row per item × language) → data/prompts/b1_pilot.csv

In [ ]:
from pathlib import Path

rows = []
for i in mt_pick:
    for lang in PILOT_LANGS:
        rows.append({"source": "mclm_math100", "item": int(i), "lang": lang, "prompt": df.loc[i, lang], "answer": df.loc[i, "answer"]})
for i in lcb_pilot.index:
    for lang in PILOT_LANGS:
        rows.append({"source": "lcb_dolly", "item": int(i), "lang": lang, "prompt": lcb_pilot.loc[i, lang], "answer": None})
pilot = pd.DataFrame(rows)

out = Path("../data/prompts/b1_pilot.csv")
out.parent.mkdir(parents=True, exist_ok=True)
pilot.to_csv(out, index=False)

back = pd.read_csv(out)                                         # round-trip check: Devanagari/Cyrillic survive
assert back["prompt"].equals(pilot["prompt"])
print(back.groupby(["source", "lang"]).size().unstack())
back.sample(5, random_state=0)

rendering the prompt

In [ ]:
from transformers import AutoTokenizer
tok = AutoTokenizer.from_pretrained("Qwen/Qwen3-0.6B")

def render(prompt, thinking):
    msgs = [{"role": "user", "content": prompt}]
    text = tok.apply_chat_template(msgs, tokenize=False,
                                   add_generation_prompt=True,
                                   enable_thinking=thinking)
    enc = tok.apply_chat_template(msgs, add_generation_prompt=True,
                                  enable_thinking=thinking,
                                  return_tensors="pt", return_dict=True)
    return text, enc

math_p = back[(back.source == "mclm_math100") & (back.lang == "ne")].iloc[0].prompt
lcb_p  = back[(back.source == "lcb_dolly")    & (back.lang == "ne")].iloc[0].prompt

for name, p, thinking in [("math / thinking ON", math_p, True),
                          ("lcb / thinking OFF", lcb_p, False)]:
    text, enc = render(p, thinking)
    print(f"===== {name} =====")
    print(text)
    print("shape:", enc["input_ids"].shape, enc["input_ids"].dtype)
    print("last 6 tokens:", tok.convert_ids_to_tokens(enc["input_ids"][0, -6:].tolist()))
    print()

print("</think> id:", tok.convert_tokens_to_ids("</think>"))